# Introduccion a la Deteccion de Patrones Algoritmicos

**Modulo:** Pattern Detection  
**Objetivo:** Aprender a usar el detector de patrones  
**Duracion estimada:** 25 minutos

---

## Contenido

1. [Setup](#setup)
2. [Arquitectura del Detector](#arquitectura-del-detector)
3. [Ejemplo Basico](#ejemplo-basico)
4. [Explorando Resultados](#explorando-resultados)
5. [Patrones Disponibles](#patrones-disponibles)
6. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import parse_pseudocode
from app.core.patterns import (
    PatternDetector,
    PatternType,
    ConfidenceLevel
)

# Para visualizacion
import pandas as pd

print("Setup completado")

---

## 2. Arquitectura del Detector

El `PatternDetector` coordina multiples detectores especificos:

```
AST -> PatternDetector -> [Detectores] -> PatternMatches -> Scorer -> Resultado
                              |
           +------------------+------------------+
           |                  |                  |
    BruteForceDetector  RecursiveDetector  DivideConquerDetector ...
```

### Patrones Soportados

| Patron | Descripcion | Complejidad Tipica |
|--------|-------------|-------------------|
| Fuerza Bruta | Exploracion exhaustiva | O(n^2) a O(2^n) |
| Recursion | Llamadas a si mismo | Variable |
| Divide y Venceras | Division y combinacion | O(n log n) |
| Programacion Dinamica | Memorizacion | O(n), O(n^2) |
| Greedy | Optimo local | O(n log n) |
| Backtracking | Prueba y retroceso | O(n!), O(2^n) |

---

## 3. Ejemplo Basico

### Detectando Patrones en Bubble Sort

In [ ]:
# Codigo de ejemplo: Bubble Sort
codigo_bubble = """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
"""

# Parsear
ast = parse_pseudocode(codigo_bubble)

# Detectar patrones
detector = PatternDetector()
result = detector.detect(ast)

# Mostrar resultados
print("RESULTADO DE DETECCION")
print(f"\nPatron principal: {result.primary_pattern_name}")
print(f"Confianza: {result.primary_confidence:.2%}")
print(f"Total patrones detectados: {result.pattern_count}")

**Salida esperada:**
```
RESULTADO DE DETECCION

Patron principal: Fuerza Bruta
Confianza: 75.00%
Total patrones detectados: 5
```

### Ejemplo: Merge Sort (Divide y Venceras)

In [ ]:
# Merge Sort - Divide y Venceras
codigo_merge = """
algorithm mergeSort(A[], left, right)
begin
    if (left < right) then
        mid <- (left + right) / 2
        call mergeSort(A, left, mid)
        call mergeSort(A, mid + 1, right)
        call merge(A, left, mid, right)
    end
end
"""

ast = parse_pseudocode(codigo_merge)
result = detector.detect(ast)

print("MERGE SORT:")
print(f"Patron principal: {result.primary_pattern_name}")
print(f"Confianza: {result.primary_confidence:.2%}")

---

## 4. Explorando Resultados

### 4.1 Todos los Patrones Detectados

In [ ]:
# Ver todos los patrones rankeados
print("TODOS LOS PATRONES DETECTADOS:")
for sp in result.all_patterns:
    pm = sp.pattern
    print(f"\n{pm.pattern_name}")
    print(f"  Confianza: {pm.confidence:.2%}")
    print(f"  Nivel: {pm.confidence_level.value}")
    print(f"  Complejidad tipica: {pm.typical_complexity}")

### 4.2 Indicadores Encontrados

In [ ]:
# Ver indicadores del patron principal
if result.primary_pattern:
    pattern = result.primary_pattern.pattern
    
    print("INDICADORES ENCONTRADOS:")
    for ind in pattern.indicators_found:
        print(f"  + {ind.name}: {ind.evidence}")
    
    print("\nINDICADORES FALTANTES:")
    for ind in pattern.indicators_missing:
        print(f"  - {ind.name}")

### 4.3 Tabla Comparativa de Patrones

In [ ]:
# Crear tabla comparativa
data = []
for sp in result.all_patterns:
    pm = sp.pattern
    data.append({
        "Patron": pm.pattern_name,
        "Confianza": f"{pm.confidence:.2%}",
        "Nivel": pm.confidence_level.value,
        "Indicadores": f"{len(pm.indicators_found)}/{pm.total_indicators}"
    })

df = pd.DataFrame(data)
print("TABLA COMPARATIVA:")
print(df.to_string(index=False))

In [ ]:
# Listar patrones disponibles
print("PATRONES DISPONIBLES:")
for pattern_name in detector.get_available_patterns():
    print(f"  - {pattern_name}")

---

## 5. Deteccion de Patron Especifico

In [ ]:
# Detectar un patron especifico
match = detector.detect_specific(ast, PatternType.RECURSIVE)

if match:
    print(f"RECURSION detectada: {match.confidence:.2%}")
else:
    print("No se detecto recursion")

In [ ]:
# Filtrar por umbral de confianza
result_high = detector.detect(ast, min_confidence=0.7)
result_low = detector.detect(ast, min_confidence=0.3)

print(f"Con umbral 0.7: {len(result_high.confident_patterns)} patron(es)")
print(f"Con umbral 0.3: {len(result_low.confident_patterns)} patron(es)")

---

## 6. Ejercicios

### Ejercicio 1: Detectar Patrones en Fibonacci

In [ ]:
# Ejercicio 1: Que patrones detecta en Fibonacci?
codigo_fib = """
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    end
    return fibonacci(n - 1) + fibonacci(n - 2)
end
"""

# Tu codigo aqui
ast_fib = parse_pseudocode(codigo_fib)
result_fib = detector.detect(ast_fib)

print(f"Patron principal: {result_fib.primary_pattern_name}")
print(f"Confianza: {result_fib.primary_confidence:.2%}")

**Pregunta:** Por que Fibonacci detecta Recursion? Que indicadores se encontraron?

### Ejercicio 2: Comparar Algoritmos

In [ ]:
# Ejercicio 2: Compara patrones de varios algoritmos
algoritmos = [
    ("Bubble Sort", codigo_bubble),
    ("Merge Sort", codigo_merge),
    ("Fibonacci", codigo_fib)
]

comparacion = []
for nombre, codigo in algoritmos:
    ast = parse_pseudocode(codigo)
    result = detector.detect(ast)
    comparacion.append({
        "Algoritmo": nombre,
        "Patron": result.primary_pattern_name,
        "Confianza": f"{result.primary_confidence:.2%}"
    })

df = pd.DataFrame(comparacion)
print("COMPARACION DE PATRONES:")
print(df.to_string(index=False))

---

## 7. Tips y Resumen

### Tips

- **Umbral de confianza:** Usa 0.3-0.4 para exploracion, 0.7+ para resultados confiables
- **Multiples patrones:** Un algoritmo puede tener varios patrones (ej: Recursion + Divide y Venceras)
- **Indicadores:** Revisa los indicadores para entender por que se detecto un patron
- **Conflictos:** Algunos patrones son mutuamente excluyentes (Fuerza Bruta vs Programacion Dinamica)

### Resumen

Has aprendido:
- Como funciona el detector de patrones
- Detectar patrones en algoritmos
- Explorar indicadores y evidencias
- Comparar patrones entre algoritmos
- Usar umbrales de confianza

---

## Proximos Pasos

- **pattern_signatures.ipynb**: Estudiar firmas de patrones
- **scoring_calibration.ipynb**: Calibrar el sistema de scoring

**Siguiente modulo recomendado**: `04_data_structures/` para detectar estructuras de datos